# استماع أعمى: أي محرك يقص الآيات أحسن؟ · Blind listening: which engine cuts ayahs best?

لكل آية تسمع ثلاث قصّات من نفس ملف الصوت: محرك عثمان، ومنجّم (whisperx)، وتوقيت mp3quran.
تظهر بأسماء «أ، ب، ج» بترتيب عشوائي، وتنكشف بعد ما تختار. كل صوت يُحفظ فوراً في Drive (`votes.json`).
For every ayah you hear three cuts of the same audio, labelled A/B/C in random order, revealed after you vote.

- يكفي وقت التشغيل العادي بالمعالج (CPU). يقرأ ملفات النوتبوكين الآخرين من Drive، فشغّلهم قبله.
- الصوت يُشغَّل مباشرة من رابط mp3quran، فيحتاج إنترنت. أول تشغيل في سورة طويلة قد يتأخر ثانية.
- **إذا انقطعت الجلسة:** Run all من جديد. الآيات التي صوّتّ عليها لا ترجع.
- الآيات المختلف عليها (فرق أكثر من `THRESHOLD` ثانية بين المصادر) تطلع أولاً، بعدها عينة عشوائية من الباقي.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/othman-timings"
RECITERS = ""      # فاضي = الكل، أو مثلاً "abdulrahman-sudais_r1, maher-muaiqly_r3"
SURAHS = ""        # فاضي = الكل، أو مثلاً "67,78"
THRESHOLD = 0.5    # أقل فرق بالثواني يخلي الآية «مختلف عليها»
RANDOM_SHARE = 0.2 # نسبة العينة العشوائية من الآيات المتقاربة

In [ ]:
import os
import sys
from google.colab import drive, userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


drive.mount("/content/drive")
BRANCH = "main"
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!rm -rf /content/othman-engine && git clone -q --depth 1 -b {BRANCH} {repo} /content/othman-engine
sys.path.insert(0, "/content/othman-engine/notebooks")
import timings

## الصفحة · The page

In [ ]:
from pathlib import Path

from google.colab import output
from IPython.display import HTML, display

votes_path = Path(OUT_DIR) / "votes.json"
votes = timings.load_votes(votes_path)
queue = timings.listen_queue(OUT_DIR, RECITERS, SURAHS, THRESHOLD, RANDOM_SHARE)
by_id = {item["id"]: item for item in queue}
pending = [item for item in queue if item["id"] not in votes]


def save_vote(item_id, choice, order):
    timings.record_vote(votes_path, votes, by_id[item_id], choice, list(order))


output.register_callback("othman.vote", save_vote)
disputed = sum(item["bucket"] == "disputed" for item in queue)
print(f"{len(queue)} ayahs ({disputed} disputed, {len(queue) - disputed} random), "
      f"{len(queue) - len(pending)} voted before, {len(pending)} to go")
if not queue:
    print("ما لقيت آيات. تأكد إن نوتبوك محرك عثمان ونوتبوك منجّم (whisperx) خلصوا على نفس السور.")
display(HTML(timings.listen_page(pending)))

## النتيجة · Results
شغّلها متى ما تبي (حتى قبل ما تخلص). «متساوية» لها سطر لحالها، ونسبة الفوز من كل الأصوات.

In [ ]:
import pandas as pd

votes = timings.load_votes(votes_path)
table = pd.DataFrame(votes.values())
if table.empty:
    raise SystemExit("ما فيه أصوات بعد · No votes yet")
names = {"othman-engine": "محرك عثمان", "munajjam-whisperx": "منجّم whisperx", "mp3quran": "mp3quran", "equal": "متساوية"}
table["winner"] = table["choice"].map(names)
print(f"{len(table)} votes")
overall = (table["winner"].value_counts(normalize=True) * 100).round(1).rename("% of votes").to_frame()
overall["votes"] = table["winner"].value_counts()
display(overall)
for column, title in (("reciter", "% per reciter"), ("bucket", "% disputed / random")):
    display((pd.crosstab(table[column], table["winner"], normalize="index") * 100).round(1).rename_axis(columns=title))